# Descriptive Statistics — Price Intelligence Platform
Summarises scraped price data: central tendency, dispersion, distribution shape.

In [ ]:
# papermill injects values here
run_date = "2024-01-01"

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

print(f"Run date : {run_date}")
print(f"NumPy    : {np.__version__}")
print(f"Pandas   : {pd.__version__}")

In [ ]:
# ---------------------------------------------------------------------------
# Load data — try Bigtable first, fall back to local JSONL exports
# ---------------------------------------------------------------------------
def _load_from_jsonl(data_dir: str = "/opt/airflow/data") -> pd.DataFrame:
    frames = []
    for jsonl in Path(data_dir).rglob("*.jsonl"):
        frames.append(pd.read_json(jsonl, lines=True))
    if not frames:
        raise FileNotFoundError(f"No JSONL files found under {data_dir}")
    return pd.concat(frames, ignore_index=True)


try:
    sys.path.insert(0, "/opt/airflow")
    from bigtable.client import BigtableClient

    client = BigtableClient(
        project_id=os.environ.get("GCP_PROJECT_ID", "local"),
        instance_id=os.environ.get("BIGTABLE_INSTANCE_ID", "price-intelligence"),
    )
    rows = client._table.read_rows(limit=5000)
    records = [client._row_to_dict(r) for r in rows]
    df = pd.DataFrame(records)
    print(f"Loaded {len(df):,} rows from Bigtable")
except Exception as exc:
    print(f"Bigtable unavailable ({exc}) — loading from JSONL")
    df = _load_from_jsonl()
    print(f"Loaded {len(df):,} rows from JSONL files")

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["price"])
df.head()

In [ ]:
# ---------------------------------------------------------------------------
# 1. Summary statistics per source
# ---------------------------------------------------------------------------
summary = (
    df.groupby("source")["price"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
        max="max",
    )
    .round(2)
)
print("\n=== Price summary per source ===")
print(summary.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 2. Price distribution — box plot per source
# ---------------------------------------------------------------------------
fig = px.box(
    df,
    x="source",
    y="price",
    color="source",
    title=f"Price Distribution by Source ({run_date})",
    labels={"price": "Price", "source": "Source"},
    log_y=True,
)
fig.show()

In [ ]:
# ---------------------------------------------------------------------------
# 3. Skewness & kurtosis per source
# ---------------------------------------------------------------------------
shape = (
    df.groupby("source")["price"]
    .agg(
        skewness=lambda s: stats.skew(s.dropna()),
        kurtosis=lambda s: stats.kurtosis(s.dropna()),
    )
    .round(3)
)
print("\n=== Distribution shape ===")
print(shape.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 4. Normality test (Shapiro-Wilk, sample ≤ 5 000)
# ---------------------------------------------------------------------------
print("\n=== Shapiro-Wilk normality test (H0: data is normal) ===")
for src, grp in df.groupby("source"):
    sample = grp["price"].dropna().head(5000)
    if len(sample) < 3:
        continue
    stat, p = stats.shapiro(sample)
    verdict = "REJECT H0 (not normal)" if p < 0.05 else "fail to reject H0"
    print(f"  {src:30s}  W={stat:.4f}  p={p:.4e}  → {verdict}")

In [ ]:
# ---------------------------------------------------------------------------
# 5. Price histogram — overall
# ---------------------------------------------------------------------------
fig2 = px.histogram(
    df,
    x="price",
    color="source",
    nbins=60,
    barmode="overlay",
    opacity=0.6,
    log_x=True,
    title=f"Price Histogram (log scale) — {run_date}",
)
fig2.show()

print("\n✓ Descriptive stats notebook complete")